# Research 22 — Cross-seed balanced-weight probability average

**Hypothesis:** averaging the two OOF probability sets from the same balanced-weight LightGBM configuration but different CV partitions may reduce split-specific noise. Compare a fixed 50/50 candidate average, then add it to the current reference at a fixed 20% weight. The component predictions are both OOF for every row; this is a synthesis screen on training OOF only, not independent validation. No weight search and no Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and np.all(probs >= 0), filename
    return probs / probs.sum(axis=1, keepdims=True)


p_base = load_probs("segment-oof.csv")
p_seed42 = load_probs("segment-lightgbm-balanced-weights-oof.csv")
p_seed2025 = load_probs("segment-lightgbm-balanced-weights-seed-2025-oof.csv")
p_crossseed = 0.5 * p_seed42 + 0.5 * p_seed2025
candidates = {
    "Current reference": p_base,
    "80/20 with balanced-weight seed 42": 0.8 * p_base + 0.2 * p_seed42,
    "80/20 with balanced-weight seed 2025": 0.8 * p_base + 0.2 * p_seed2025,
    "80/20 with 50/50 cross-seed average": 0.8 * p_base + 0.2 * p_crossseed,
}
fold_splits = {
    "seed42": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y)
    ),
    "seed2025": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=2025).split(np.zeros(len(y)), y)
    ),
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed_name, folds in fold_splits.items():
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            row[f"{seed_name}_fold_{fold}_macro_f1"] = f1_score(
                y[valid_idx], pred[valid_idx], average="macro"
            )
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-lightgbm-balanced-weights-crossseed.csv", index=False)
pd.DataFrame(
    candidates["80/20 with 50/50 cross-seed average"], columns=[f"prob_{i}" for i in range(4)]
).assign(segment=y).to_csv(
    ROOT / "predictions/segment-lightgbm-balanced-weights-crossseed-blend-oof.csv", index=False
)
print("OOF synthesis complete; both component OOF files align to train order. Test not loaded.")

,candidate,macro_f1,weighted_f1,seed42_fold_1_macro_f1,seed42_fold_2_macro_f1,seed42_fold_3_macro_f1,seed2025_fold_1_macro_f1,seed2025_fold_2_macro_f1,seed2025_fold_3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462
1,80/20 with balanced-weight seed 42,0.751550,0.764986,0.749726,0.750842,0.754051,0.750878,0.750670,0.753089
2,80/20 with balanced-weight seed 2025,0.751429,0.764984,0.749780,0.750280,0.754196,0.751178,0.750049,0.753047
3,80/20 with 50/50 cross-seed average,0.751378,0.764927,0.749626,0.750884,0.753590,0.751349,0.750419,0.752357


OOF synthesis complete; both component OOF files align to train order. Test not loaded.


## Interpretation limits

All three mixture recipes are fixed before scoring. Evaluating them repeatedly on the same labels/folds can overstate selection performance; the cross-seed average is a stability hypothesis, not proof of generalization.